In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

## 1. Load dataset

In [3]:
df = pd.read_csv('data/student.csv')
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [4]:
X = df.drop(columns=['math_score'])
y = df['math_score']

## 2. Train test split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3)

In [6]:
X_train.columns

Index(['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch',
       'test_preparation_course', 'reading_score', 'writing_score'],
      dtype='str')

## 3. Feature groups

In [7]:
numeric_features = X_train.select_dtypes(exclude=['object', 'string']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns.tolist()
print(f"Numeric features: {numeric_features}")
print(f"categorical features: {categorical_features}")

Numeric features: ['reading_score', 'writing_score']
categorical features: ['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course']


In [8]:
X_train['parental_level_of_education'].unique()

<StringArray>
[  'some high school',       'some college',        'high school',
  'bachelor's degree', 'associate's degree',    'master's degree']
Length: 6, dtype: str

In [9]:
ordinal_features = ['parental_level_of_education']
education_order = ['some high school', 'high school', 'some college', "associate's degree", "bachelor's degree", "master's degree"]
nominal_features = [col for col in categorical_features if col not in ordinal_features]

In [10]:
print(f"Nominal features: {nominal_features}")

Nominal features: ['gender', 'race_ethnicity', 'lunch', 'test_preparation_course']


## 4. Pipelines per feature type

In [11]:
numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler())])
nominal_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown="ignore", drop='first', sparse_output=True))])
ordinal_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OrdinalEncoder(categories=[education_order], handle_unknown="use_encoded_value", unknown_value=-1))])

In [12]:
preprocessor = ColumnTransformer([('numeric', numeric_pipeline, numeric_features), ('nominal', nominal_pipeline, nominal_features), ('ordinal', ordinal_pipeline, ordinal_features)])

## 5. Full pipeline with model

In [13]:
model = LinearRegression()

model_pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])

## 6. Fit the model

In [14]:
model_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('nominal', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 7. Model evaluation on test set

In [15]:
def evaluate_model(X, y):
    y_pred = model_pipeline.predict(X)
    mse = mean_squared_error(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    return mse, mae, r2

## 8. Testing various models

In [16]:
models = {'Linear Regression': LinearRegression(),
'Lasso': Lasso(),
'Ridge': Ridge(),
'Decision Tree': DecisionTreeRegressor(),
'Random Forest': RandomForestRegressor(),
'Ada Boost': AdaBoostRegressor(),
'K Neighbors': KNeighborsRegressor(),
'SVR': SVR(),
'Catboost': CatBoostRegressor(),
'Xgboost': XGBRegressor()}

In [17]:
eval_metrics = pd.DataFrame(np.nan, index=models.keys(), columns=['train_mse', 'train_mae', 'train_r2', 'test_mse', 'test_mae', 'test_r2'])

In [18]:
for k, v in models.items():
    model_pipeline = Pipeline([('preprocessing', preprocessor), ('model', v)])
    model_pipeline.fit(X_train, y_train)
    print(f"Training Model: {k}\n")
    mse, mae, r2 = evaluate_model(X_train, y_train)
    eval_metrics.loc[k, 'train_mse'] = mse
    eval_metrics.loc[k, 'train_mae'] = mae
    eval_metrics.loc[k, 'train_r2'] = r2

    mse, mae, r2 = evaluate_model(X_test, y_test)
    eval_metrics.loc[k, 'test_mse'] = mse
    eval_metrics.loc[k, 'test_mae'] = mae
    eval_metrics.loc[k, 'test_r2'] = r2

Training Model: Linear Regression



Training Model: Lasso

Training Model: Ridge

Training Model: Decision Tree

Training Model: Random Forest

Training Model: Ada Boost

Training Model: K Neighbors

Training Model: SVR

Learning rate set to 0.039525
0:	learn: 14.9429672	total: 128ms	remaining: 2m 8s
1:	learn: 14.5385154	total: 129ms	remaining: 1m 4s
2:	learn: 14.1766416	total: 130ms	remaining: 43.1s
3:	learn: 13.8996662	total: 130ms	remaining: 32.4s
4:	learn: 13.5223696	total: 131ms	remaining: 26s
5:	learn: 13.1907559	total: 131ms	remaining: 21.7s
6:	learn: 12.9654438	total: 131ms	remaining: 18.6s
7:	learn: 12.6702403	total: 132ms	remaining: 16.4s
8:	learn: 12.3548606	total: 132ms	remaining: 14.6s
9:	learn: 12.0419582	total: 133ms	remaining: 13.2s
10:	learn: 11.7665549	total: 133ms	remaining: 12s
11:	learn: 11.4924477	total: 134ms	remaining: 11s
12:	learn: 11.2335176	total: 135ms	remaining: 10.2s
13:	learn: 10.9957379	total: 135ms	remaining: 9.51s
14:	learn: 10.7456924	total: 136ms	remaining: 8.9s
15:	learn: 10.5114318	

In [19]:
eval_metrics.sort_values(by='test_r2', ascending=False)

,train_mse,train_mae,train_r2,test_mse,test_mae,test_r2
Ridge,27.808008,4.193683,0.882193,31.173279,4.473882,0.847351
Linear Regression,27.802890,4.194622,0.882215,31.229441,4.474701,0.847076
Random Forest,5.065172,1.797051,0.978542,38.853560,4.909846,0.809742
Ada Boost,32.435798,4.615912,0.862588,38.878542,4.957756,0.809619
Catboost,9.269073,2.358293,0.960732,39.701589,5.107573,0.805589
SVR,48.279530,5.193324,0.795467,42.236121,5.045050,0.793178
Lasso,42.719545,5.173179,0.819022,43.355863,5.159657,0.787695
Xgboost,0.889402,0.624058,0.996232,44.862415,5.359961,0.780317
K Neighbors,33.392100,4.570500,0.858537,48.987400,5.651000,0.760118
Decision Tree,0.015625,0.006250,0.999934,71.055000,6.855000,0.652057
